# PySpark: Basic Introduction + DataFrames (Part 1)
Combined notebook. **Part 1** covers setup and reading a CSV; **Part 2** covers common DataFrame operations.

Data file used throughout: `test1.csv` (keep it in the same folder as this notebook).

In [8]:
from pyspark.sql import SparkSession
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Part 1 - PySpark Basic Introduction

In [9]:
# CELL 1: Install PySpark
# "!" runs a shell command from inside Jupyter. pip installs the pyspark package
# (and its dependency py4j, which lets Python talk to the Java-based Spark engine).
# If PySpark is already installed, pip just reports "Requirement already satisfied".
!pip install pyspark

In [10]:
# CELL 2: Import the pyspark package
# A quick sanity check - if this runs without error, PySpark is installed correctly.
import pyspark

In [11]:
# CELL 3: Compare with Pandas
# Reads the same CSV with pandas and prints the type of the result.
# Output is pandas.core.frame.DataFrame -> a pandas DataFrame lives in one machine's memory.
# A PySpark DataFrame (seen later) is distributed and evaluated lazily.
import pandas as pd
type(pd.read_csv('test1.csv'))

pandas.core.frame.DataFrame

In [12]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
# CELL 4: Import SparkSession
# SparkSession is the single entry point for working with DataFrames in PySpark.
from pyspark.sql import SparkSession

In [14]:
# CELL 5: Create (or reuse) a Spark session
# builder            -> starts configuring a session
# .appName('Practise') -> a name for this application (shows up in the Spark UI)
# .getOrCreate()     -> returns the existing session if one is running, else creates a new one
spark = SparkSession.builder.appName('Practise').getOrCreate()

In [15]:
# CELL 6: Inspect the session
# Displays session details: Spark version, master (local[*] = run locally using all CPU cores),
# app name, and a link to the Spark UI for monitoring jobs.
spark

In [16]:
# CELL 7: Read a CSV WITHOUT options
# spark.read.csv() loads the file into a PySpark DataFrame.
# Without any options, the header row is treated as normal data and
# all columns are named _c0, _c1, ... (this is why the next cell adds the header option).
df_pyspark = spark.read.csv('test1.csv')

In [17]:
# CELL 8: Read a CSV WITH the header option
# option('header', 'true') tells Spark that the first row holds the column names.
# Note: without inferSchema, every column is read as a string.
df_pyspark = spark.read.option('header', 'true').csv('test1.csv')

In [18]:
# CELL 9: Check the object type
# Confirms this is a pyspark.sql.dataframe.DataFrame (not a pandas DataFrame).
type(df_pyspark)

pyspark.sql.classic.dataframe.DataFrame

In [19]:
# CELL 10: Print the schema (column names + data types)
# Because inferSchema was NOT used in Cell 8, every column shows as "string",
# including numeric columns like age. Part 2 fixes this with inferSchema=True.
df_pyspark.printSchema()

root
 |-- Name: string (nullable = true)
 |-- age: string (nullable = true)
 |-- Experience: string (nullable = true)
 |-- Salary: string (nullable = true)



## Part 2 - PySpark DataFrames (Part 1)
Topics covered:
- PySpark DataFrame
- Reading the dataset
- Checking the datatypes of the columns (schema)
- Selecting columns and indexing
- `describe()` (similar to Pandas)
- Adding columns
- Dropping columns
- Renaming columns

In [20]:
# CELL 11: Import SparkSession (again)
# Harmless repeat from Part 1 - kept so Part 2 can also run on its own.
from pyspark.sql import SparkSession

In [21]:
# CELL 12: Create (or reuse) a Spark session
# Since a session already exists from Part 1, getOrCreate() simply returns it,
# so the app name stays 'Practise'. Restart the kernel and run from here
# if you want a fresh session named 'Dataframe'.
spark = SparkSession.builder.appName('Dataframe').getOrCreate()

In [22]:
# CELL 13: Inspect the session
# Shows Spark version, master, app name and the Spark UI link.
spark

In [23]:
# CELL 14: Read the dataset with header + schema inference
# option('header','true') -> first row = column names
# inferSchema=True        -> Spark scans the data and picks proper types (e.g. int instead of string)
df_pyspark = spark.read.option('header', 'true').csv('test1.csv', inferSchema=True)

In [24]:
# CELL 15: Check the schema
# Now numeric columns (age, Experience, Salary) appear as integer instead of string.
df_pyspark.printSchema()

root
 |-- Name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- Experience: integer (nullable = true)
 |-- Salary: integer (nullable = true)



In [25]:
# CELL 16: Alternative, shorter way to read the CSV
# header and inferSchema are passed directly as arguments to csv().
# show() prints the DataFrame as a table (first 20 rows by default).
df_pyspark = spark.read.csv('test1.csv', header=True, inferSchema=True)
df_pyspark.show()

+---------+---+----------+------+
|     Name|age|Experience|Salary|
+---------+---+----------+------+
|    Krish| 31|        10| 30000|
|Sudhanshu| 30|         8| 25000|
|    Sunny| 29|         4| 20000|
|     Paul| 24|         3| 20000|
|   Harsha| 21|         1| 15000|
|  Shubham| 23|         2| 18000|
+---------+---+----------+------+



In [26]:
# CELL 17: Check the schema again
# Same result as Cell 15 - confirms both reading styles are equivalent.
df_pyspark.printSchema()

root
 |-- Name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- Experience: integer (nullable = true)
 |-- Salary: integer (nullable = true)



In [27]:
# CELL 18: Check the type
# It is a pyspark.sql.dataframe.DataFrame.
type(df_pyspark)

pyspark.sql.classic.dataframe.DataFrame

In [28]:
# CELL 19: head(n)
# Returns the first n rows as a Python list of Row objects
# (unlike show(), which just prints a table).
df_pyspark.head(3)

[Row(Name='Krish', age=31, Experience=10, Salary=30000),
 Row(Name='Sudhanshu', age=30, Experience=8, Salary=25000),
 Row(Name='Sunny', age=29, Experience=4, Salary=20000)]

In [29]:
# CELL 20: show()
# Prints the DataFrame in a readable table format.
df_pyspark.show()

+---------+---+----------+------+
|     Name|age|Experience|Salary|
+---------+---+----------+------+
|    Krish| 31|        10| 30000|
|Sudhanshu| 30|         8| 25000|
|    Sunny| 29|         4| 20000|
|     Paul| 24|         3| 20000|
|   Harsha| 21|         1| 15000|
|  Shubham| 23|         2| 18000|
+---------+---+----------+------+



In [30]:
# CELL 21: Select multiple columns
# select() takes a list of column names and returns a NEW DataFrame with only those columns.
df_pyspark.select(['Name', 'Experience']).show()

+---------+----------+
|     Name|Experience|
+---------+----------+
|    Krish|        10|
|Sudhanshu|         8|
|    Sunny|         4|
|     Paul|         3|
|   Harsha|         1|
|  Shubham|         2|
+---------+----------+



In [31]:
# CELL 22: Select a single column using indexing
# df['Name'] returns a Column object (not the data itself).
# Use select('Name') if you want a DataFrame you can show().
df_pyspark['Name']

Column<'Name'>

In [32]:
# CELL 23: dtypes
# Returns a list of (column_name, data_type) tuples - a quick way to see each column's type.
df_pyspark.dtypes

[('Name', 'string'), ('age', 'int'), ('Experience', 'int'), ('Salary', 'int')]

In [33]:
# CELL 24: describe()
# Summary statistics like pandas: count, mean, stddev, min, max.
# For string columns (Name) mean/stddev are null; min/max are alphabetical.
df_pyspark.describe().show()

+-------+------+------------------+-----------------+------------------+
|summary|  Name|               age|       Experience|            Salary|
+-------+------+------------------+-----------------+------------------+
|  count|     6|                 6|                6|                 6|
|   mean|  NULL|26.333333333333332|4.666666666666667|21333.333333333332|
| stddev|  NULL| 4.179314138308661|3.559026084010437| 5354.126134736337|
|    min|Harsha|                21|                1|             15000|
|    max| Sunny|                31|               10|             30000|
+-------+------+------------------+-----------------+------------------+



In [34]:
# CELL 25: Add a new column
# withColumn(new_name, expression) returns a new DataFrame with the extra column.
# Here: Experience + 2 -> experience after two more years.
# DataFrames are immutable, so the result is assigned back to df_pyspark.
df_pyspark = df_pyspark.withColumn('Experience After 2 year', df_pyspark['Experience'] + 2)

In [35]:
# CELL 26: View the DataFrame with the new column
df_pyspark.show()

+---------+---+----------+------+-----------------------+
|     Name|age|Experience|Salary|Experience After 2 year|
+---------+---+----------+------+-----------------------+
|    Krish| 31|        10| 30000|                     12|
|Sudhanshu| 30|         8| 25000|                     10|
|    Sunny| 29|         4| 20000|                      6|
|     Paul| 24|         3| 20000|                      5|
|   Harsha| 21|         1| 15000|                      3|
|  Shubham| 23|         2| 18000|                      4|
+---------+---+----------+------+-----------------------+



In [36]:
# CELL 27: Drop a column
# drop() removes the named column and returns a new DataFrame.
# The result is assigned back so the extra column is gone from df_pyspark.
df_pyspark = df_pyspark.drop('Experience After 2 year')

In [37]:
# CELL 28: Confirm the column was dropped
df_pyspark.show()

+---------+---+----------+------+
|     Name|age|Experience|Salary|
+---------+---+----------+------+
|    Krish| 31|        10| 30000|
|Sudhanshu| 30|         8| 25000|
|    Sunny| 29|         4| 20000|
|     Paul| 24|         3| 20000|
|   Harsha| 21|         1| 15000|
|  Shubham| 23|         2| 18000|
+---------+---+----------+------+



In [38]:
# CELL 29: Rename a column
# withColumnRenamed(old_name, new_name) returns a new DataFrame with the column renamed.
# The result is NOT assigned back, so df_pyspark itself keeps the original name 'Name'.
df_pyspark.withColumnRenamed('Name', 'New Name').show()

+---------+---+----------+------+
| New Name|age|Experience|Salary|
+---------+---+----------+------+
|    Krish| 31|        10| 30000|
|Sudhanshu| 30|         8| 25000|
|    Sunny| 29|         4| 20000|
|     Paul| 24|         3| 20000|
|   Harsha| 21|         1| 15000|
|  Shubham| 23|         2| 18000|
+---------+---+----------+------+

